# **Intro to Parallel Process Lab 4**
Name : Thirunavukarasu MuthuValliappan ;
Student Number : 3785346

Installing MPI Compilers and MPI Program launcher

In [1]:
!apt-get update
!apt-get install -y openmpi-bin libopenmpi-dev

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,8

MPI Code for PingPong and Ring

In [13]:
%%writefile Lab4.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define TAG_RANGE 1000
#define K_ITERATIONS 10000
#define TAG01 100
#define TAG10 101
#define TAGR 200

void run_pingpong(int rank) {
    int msg_sizes[] = {1, 8, 64, 512, 4096, 65536};
    int num_sizes = 6;

    if (rank == 0) {
        printf("%-20s | %-12s | %-15s | %-15s\n",
               "Message Size (bytes)", "Iterations", "Avg RTT (s)", "Est. Latency (s)");
        printf("----------------------------------------------------------------------------\n");
    }

    for (int s = 0; s < num_sizes; s++) {
        int msg_size = msg_sizes[s];
        char *buffer = (char *)malloc(msg_size);

        double start_time = 0.0;
        if (rank == 0) {
            start_time = MPI_Wtime();
        }

        for (int k = 0; k < K_ITERATIONS; k++) {
            int tag = k % TAG_RANGE;

            if (rank == 0) {
                // Rank 0: Send -> Receive
                MPI_Send(buffer, msg_size, MPI_BYTE, 1, tag, MPI_COMM_WORLD);
                MPI_Recv(buffer, msg_size, MPI_BYTE, 1, tag, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
            } else if (rank == 1) {
                // Rank 1: Receive -> Send
                MPI_Status status;
                MPI_Recv(buffer, msg_size, MPI_BYTE, 0, tag, MPI_COMM_WORLD, &status);

                // Verify correctness using tags, not content
                if (status.MPI_TAG != tag) {
                    printf("Tag mismatch at iteration %d\n", k);
                }
                MPI_Send(buffer, msg_size, MPI_BYTE, 0, tag, MPI_COMM_WORLD);
            }
        }

        if (rank == 0) {
            double end_time = MPI_Wtime();
            double total_time = end_time - start_time;
            double avg_rtt = total_time / K_ITERATIONS;
            double latency = avg_rtt / 2.0;

            printf("%-20d | %-12d | %-15.9f | %-15.9f\n",
                   msg_size, K_ITERATIONS, avg_rtt, latency);
        }
        free(buffer);
    }
}

void run_ring(int rank, int size) {
    int send_value = rank; // Initial value is the rank
    int recv_value = -1;
    int right = (rank + 1) % size; // Right neighbour logic
    int left = (rank - 1 + size) % size; // Left neighbour logic

    // Use MPI_Sendrecv to avoid deadlock
    MPI_Sendrecv(&send_value, 1, MPI_INT, right, TAGR,
                 &recv_value, 1, MPI_INT, left, TAGR,
                 MPI_COMM_WORLD, MPI_STATUS_IGNORE);

    // Confirm the exchange
    printf("Rank %d: sent %d to right, received %d from left\n",
           rank, send_value, recv_value);
}

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (argc < 2) {
        if (rank == 0) printf("Usage: %s pingpong|ring\n", argv[0]);
        MPI_Finalize();
        return 0;
    }

    if (strcmp(argv[1], "pingpong") == 0) {
        if (size != 2) {
            if (rank == 0) printf("Ping-pong requires -np 2\n");
        } else {
            run_pingpong(rank);
        }
    }
    else if (strcmp(argv[1], "ring") == 0) {
        if (size < 2) {
            if (rank == 0) printf("Ring requires -np >= 2\n");
        } else {
            run_ring(rank, size);
        }
    }

    MPI_Finalize();
    return 0;
}

Overwriting Lab4.c


Compilation

In [14]:
!mpicc Lab4.c -o Lab4

PingPong Execution

In [15]:
!mpirun --allow-run-as-root --oversubscribe -np 2 ./Lab4 pingpong

Message Size (bytes) | Iterations   | Avg RTT (s)     | Est. Latency (s)
----------------------------------------------------------------------------
1                    | 10000        | 0.000004717     | 0.000002359    
8                    | 10000        | 0.000004433     | 0.000002216    
64                   | 10000        | 0.000002834     | 0.000001417    
512                  | 10000        | 0.000003652     | 0.000001826    
4096                 | 10000        | 0.000013162     | 0.000006581    
65536                | 10000        | 0.000029190     | 0.000014595    


Ring Execution

In [16]:
!mpirun --allow-run-as-root --oversubscribe -np 8 ./Lab4 ring

Rank 0: sent 0 to right, received 7 from left
Rank 7: sent 7 to right, received 6 from left
Rank 5: sent 5 to right, received 4 from left
Rank 6: sent 6 to right, received 5 from left
Rank 1: sent 1 to right, received 0 from left
Rank 2: sent 2 to right, received 1 from left
Rank 4: sent 4 to right, received 3 from left
Rank 3: sent 3 to right, received 2 from left
